### Imports and shared pipeline
Sets up import access to the project's `src/` package (added to `sys.path` since this is a plain script-style `uv` project, not an installed package), then imports the shared preprocessing pipeline (`get_processed_data`, `get_cv_splitter`) plus the sklearn pieces used for modeling and evaluation. Using `src.preprocessing.get_processed_data()` instead of loading/cleaning/encoding inline means this notebook gets the exact same final dataset as every other notebook and every teammate's machine.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, root_mean_squared_error

from src.preprocessing import get_processed_data, get_cv_splitter

### Get the final, cleaned & encoded dataset
One call to the shared pipeline in `src/preprocessing.py`: reads `train.csv`/`test.csv`, drops the known `ID=1770` outlier row from training before the split (`src/data.py`), does the 80/20 train/val split (`random_state=42`), drops constant/duplicate columns computed from `X_train` only, and ordinal-encodes the categorical columns (`handle_unknown='use_encoded_value'`, safe for the unseen `X0`/`X2`/`X5` test categories) fit on `X_train` only. Returns a `ProcessedData` object — `data.X_train`, `data.X_val`, `data.X_test`, `data.y_train`, `data.y_val`, `data.test_ID` — ready to hand straight to a model.

In [2]:
data = get_processed_data()

data.X_train

,cat__X0,cat__X1,cat__X2,cat__X3,cat__X4,cat__X5,cat__X6,cat__X8,num__X10,num__X12,...,num__X373,num__X374,num__X375,num__X376,num__X377,num__X378,num__X379,num__X380,num__X383,num__X384
3829,35.0,13.0,3.0,5.0,3.0,25.0,0.0,1.0,0,0,...,0,1,0,0,0,0,0,0,0,0
152,31.0,1.0,26.0,3.0,3.0,13.0,8.0,4.0,0,0,...,0,0,0,1,0,0,0,0,0,0
1556,19.0,10.0,16.0,2.0,3.0,3.0,11.0,10.0,0,0,...,0,0,0,0,1,0,0,0,0,0
2357,18.0,23.0,8.0,5.0,3.0,18.0,6.0,5.0,0,1,...,1,0,0,0,0,0,0,0,0,0
1315,36.0,22.0,40.0,2.0,3.0,2.0,4.0,5.0,0,0,...,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3444,19.0,19.0,37.0,5.0,3.0,22.0,11.0,6.0,0,1,...,0,1,0,0,0,0,0,0,0,0
466,20.0,25.0,24.0,2.0,3.0,9.0,9.0,9.0,0,0,...,0,0,0,0,0,0,0,0,0,0
3092,43.0,14.0,16.0,5.0,3.0,21.0,6.0,11.0,0,0,...,0,0,1,0,0,0,0,0,0,0
3772,43.0,20.0,16.0,2.0,3.0,25.0,6.0,0.0,0,0,...,0,0,1,0,0,0,0,0,0,0


### Dummy Baseline

In [3]:
kfold = get_cv_splitter()

dummy_model = DummyRegressor(strategy='mean')
dummy_scores = cross_val_score(dummy_model, data.X_train, data.y_train, cv=kfold, scoring='r2')

print(f"Dummy R2 per fold: {dummy_scores}")
print(f"Dummy R2 mean:     {dummy_scores.mean():.4f}")

Dummy R2 per fold: [-1.46684746e-03 -1.55261768e-06 -1.68868821e-03 -7.32943901e-05
 -1.00077081e-04]
Dummy R2 mean:     -0.0007


### Baseline model
Starting as simple as possible: plain `LinearRegression` (ordinary least squares — no regularization, no tuning) on the shared dataset from `src.preprocessing`. The point of a baseline is an honest, simple starting score to improve on — later notebooks (`03_model_comparison.ipynb`) try more complex models against this same dataset, so any improvement they show is a real improvement, not just a different preprocessing setup.

No scaling applied: for unregularized OLS, predictions are (up to floating-point noise) invariant to per-feature scaling — verified on this dataset, scaled vs. unscaled validation R2 differ by ~0.0007, not a real effect. Not worth the extra pipeline step for a "start simple" baseline.

In [4]:
baseline_model = LinearRegression()
baseline_model.fit(data.X_train, data.y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](310,)","[ 0.06,-0.04,-0.03,..., 1. ,14.96, 0.42]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](310,)","['cat__X0','cat__X1','cat__X2',...,'num__X380','num__X383','num__X384']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,113
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,310
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(251)


### Evaluate on validation set
Predicts on `X_val`, which was never used to fit the encoder or the model. R2 is the competition's actual scoring metric; RMSE is a more intuitive "average miss" in `y`'s own units, kept as a secondary sanity check.

In [5]:
y_val_pred = baseline_model.predict(data.X_val)

r2 = r2_score(data.y_val, y_val_pred)
rmse = root_mean_squared_error(data.y_val, y_val_pred)

print(f"Validation R2:   {r2:.4f}")
print(f"Validation RMSE: {rmse:.4f}")

Validation R2:   0.5131
Validation RMSE: 8.7404


## Findings so far / what should inform next steps

**Baseline result**
- Plain `LinearRegression` (no regularization, no tuning) on the shared, cleaned, encoded dataset from `src.preprocessing.get_processed_data()` — see Validation R2/RMSE above. Public leaderboard scores for this competition typically top out around 0.55-0.58, so this is a believable, unremarkable starting point — the goal now is to see how much (if any) more complex models actually improve on it, not to treat this number as final.

**Feature structure**
- 8 categorical (text) columns: `X0, X1, X2, X3, X4, X5, X6, X8`. Cardinality ranges from 4 (`X4`) to 47-49 (`X0`).
- 368 numeric columns, all binary (0/1).
- `X0`, `X2`, `X5` have categories in `test.csv` that never appear in `train.csv` — this is why ordinal encoding with `unknown_value=-1` is used instead of one-hot, fit on `X_train` only (`src/preprocessing.py::build_preprocessor`). Note this same ordinal encoding is what all models built on this dataset see, including linear ones — a category coded `5` isn't "half of" category `10`, but `LinearRegression` has no way to know that. Worth remembering if a model's coefficients look odd for a categorical column.

**Known outlier — now handled centrally**
- Training row `ID=1770` (`y=265.32`, ~13 std devs above the mean) is dropped from `df_train` before the train/val split, inside `src/data.py::load_raw_data`. It's out of both `X_train` and `X_val` now (not just training) — one place, so every notebook and every teammate gets the same outlier-free data automatically.

**Column cleanup — now handled centrally**
- Constant and exact-duplicate columns (computed from `X_train` only) are dropped automatically inside `get_processed_data()` (`src/preprocessing.py::find_columns_to_drop`). Constant columns carry no signal; duplicate columns are exact multicollinearity — dropping both is safe and reduces noise regardless of which model is used downstream.

**Still open**
- Check the train/val R2 gap below — a large gap would mean the model is overfitting (memorizing training-specific patterns rather than generalizing), which for a plain linear model on this many features (particularly with the ordinal-encoded categoricals) is worth watching even without an ensemble model's flexibility.
- Any change should be compared against the R2 above before being kept.

### Train vs. validation R2 gap
How much the model's R2 on data it trained on (`X_train`) exceeds its R2 on data it never saw (`X_val`) — a large gap is a classic overfitting signature: the model is fitting patterns specific to the training rows rather than signal that generalizes.

In [6]:
train_r2 = r2_score(data.y_train, baseline_model.predict(data.X_train))
val_r2   = r2_score(data.y_val, baseline_model.predict(data.X_val))

print(f"Train R2:      {train_r2:.4f}")
print(f"Validation R2: {val_r2:.4f}")
print(f"Gap:           {train_r2 - val_r2:.4f}")

Train R2:      0.6324
Validation R2: 0.5131
Gap:           0.1193
